In [80]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from shapely.geometry import Point
import geopandas
import matplotlib.pyplot as plt
import json
import re

In [18]:
pangloss_corupus = 'https://pangloss.cnrs.fr/corpus?lang=en'

In [39]:
with open('pangloss.html', 'r') as file:
    # Read the contents of the file
    html_content = file.read()
    # Print or process the content as needed
    print(html_content)
div_list = soup.find('div', {'id': 'list', 'class': 'list'})
links = div_list.find_all('a')

hrefs = []
languages = []

for link in links:
    href = link.get('href')
    language_name = link.text
    if href and language_name:
        hrefs.append(href)
        languages.append(language_name)
    print(f"Language: {language_name}, Href: {href}")


df = pd.DataFrame({'Href': hrefs, 'Language': languages})


<div id="list" class="list"><div class="group"><div class="title">A</div><div><a href="https://pangloss.cnrs.fr/corpus/Abzakh?lang=en" updated="">Abzakh</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Achhame?lang=en" updated="">Achhame (Nepali dialect)</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Aheu?lang=en" updated="">Aheu</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Aji%C3%AB?lang=en" updated="">Ajië</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Tib%C3%A9tain_de_l%E2%80%99Amdo?lang=en" updated="">Amdo Tibetan</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Chaoui_des_Amouchas?lang=en" updated="">Amoucha Shawiya</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Angait%C3%A9?lang=en" updated="">Angaité</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Ao?lang=en" updated="">Ao</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Araki?lang=en" updated="">Araki</a></div><div><a href="https://pangloss.cnrs.fr/corpus/Arem?lang=en" 

In [40]:
df

,Href,Language
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh
1,https://pangloss.cnrs.fr/corpus/Achhame?lang=en,Achhame (Nepali dialect)
2,https://pangloss.cnrs.fr/corpus/Aheu?lang=en,Aheu
3,https://pangloss.cnrs.fr/corpus/Aji%C3%AB?lang=en,Ajië
4,https://pangloss.cnrs.fr/corpus/Tib%C3%A9tain_...,Amdo Tibetan
...,...,...
246,https://pangloss.cnrs.fr/corpus/Yemeni_Arabic_...,Yemeni Arabic (Sanaa dialect)
247,https://pangloss.cnrs.fr/corpus/Yemeni_Arabic_...,Yemeni Arabic (Zabid dialect)
248,https://pangloss.cnrs.fr/corpus/Yongning_Na?la...,Yongning Na
249,https://pangloss.cnrs.fr/corpus/Yuanga?lang=en,Yuanga


In [100]:
def extract_urls_from_html(url):
    try:
        # Make a request to the URL
        response = requests.get(url)
        
        # Check if the request was successful
        if response.status_code == 200:
            # Parse the HTML content
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Find all <tr> tags with onclick attributes
            tr_tags = soup.find_all('tr', onclick=True)
            
            extracted_urls = []
            
            # Loop through each <tr> tag
            for tr_tag in tr_tags:
                onclick_attribute = tr_tag['onclick']
                # Extract URLs using regular expression
                urls = re.findall(r"https?://[^']+", onclick_attribute)
                extracted_urls.extend(urls)
            
            return extracted_urls
        else:
            print(f"Failed to fetch data from {url}. Status code: {response.status_code}")
            return []
    except Exception as e:
        print(f"An error occurred while processing {url}: {e}")
        return []


In [101]:
df['items_urls'] = df['Href'].apply(extract_urls_from_html)

In [132]:
df

,Href,Language,items_urls
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,[https://pangloss.cnrs.fr/corpus/show?oai_prim...
1,https://pangloss.cnrs.fr/corpus/Achhame?lang=en,Achhame (Nepali dialect),[https://pangloss.cnrs.fr/corpus/show?oai_prim...
2,https://pangloss.cnrs.fr/corpus/Aheu?lang=en,Aheu,[https://pangloss.cnrs.fr/corpus/show?oai_prim...
3,https://pangloss.cnrs.fr/corpus/Aji%C3%AB?lang=en,Ajië,[https://pangloss.cnrs.fr/corpus/show?oai_prim...
4,https://pangloss.cnrs.fr/corpus/Tib%C3%A9tain_...,Amdo Tibetan,[https://pangloss.cnrs.fr/corpus/show?oai_prim...
...,...,...,...
246,https://pangloss.cnrs.fr/corpus/Yemeni_Arabic_...,Yemeni Arabic (Sanaa dialect),[https://pangloss.cnrs.fr/corpus/show?oai_prim...
247,https://pangloss.cnrs.fr/corpus/Yemeni_Arabic_...,Yemeni Arabic (Zabid dialect),[https://pangloss.cnrs.fr/corpus/show?oai_prim...
248,https://pangloss.cnrs.fr/corpus/Yongning_Na?la...,Yongning Na,[https://pangloss.cnrs.fr/corpus/show?oai_prim...
249,https://pangloss.cnrs.fr/corpus/Yuanga?lang=en,Yuanga,[https://pangloss.cnrs.fr/corpus/show?oai_prim...


In [145]:
df_exploded = df.explode('items_urls')

In [146]:
df_exploded

,Href,Language,items_urls
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
...,...,...,...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...


In [152]:
df_exploded = df_exploded.dropna(subset = ['items_urls'])

In [153]:
df_exploded

,Href,Language,items_urls
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...
...,...,...,...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...


In [157]:
def extract_wav_urls_from_html(url):
    # Parse the HTML
    html_content = requests.get(url).text
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the <ul> tag with class "download-list"
    ul_tag = soup.find('ul', class_='download-list')

    # Initialize a list to store the URLs ending with ".wav"
    wav_urls = []

    # Check if the <ul> tag is found
    if ul_tag:
        # Find all <a> tags within the <ul> tag
        a_tags = ul_tag.find_all('a', href=True)
        
        # Iterate through each <a> tag
        for a_tag in a_tags:
            href = a_tag['href']
            print(href)
            # Check if the href ends with ".wav"
            if href.endswith('.wav'):
                wav_urls.append(href)

    return wav_urls

In [159]:
df_exploded['wav'] = df_exploded['items_urls'].apply(extract_wav_urls_from_html)

https://cocoon.huma-num.fr/data/archi/masters/125792.wav
https://cocoon.huma-num.fr/data/archi/mp3/125792_GARCON_44k.mp3
https://cocoon.huma-num.fr/data/lacito/masters/crdo-ADY_GARCON.xml
https://cocoon.huma-num.fr/data/archi/masters/125799.wav
https://cocoon.huma-num.fr/data/archi/mp3/125799_PATSITSE_44k.mp3
https://cocoon.huma-num.fr/data/lacito/masters/crdo-ADY_PATSITSE.xml
https://cocoon.huma-num.fr/data/archi/masters/125798.wav
https://cocoon.huma-num.fr/data/archi/mp3/125798_LISTE_NIAZ_44k.mp3
https://cocoon.huma-num.fr/data/lacito/masters/crdo-ADY_LISTE_NIAZ.xml
https://cocoon.huma-num.fr/data/konuk/masters/ADY_MK_NARR_001.wav
https://cocoon.huma-num.fr/data/konuk/ADY_MK_NARR_001.mp3
https://cocoon.huma-num.fr/data/konuk/ADY_MK_NARR_001.xml
https://cocoon.huma-num.fr/data/konuk/masters/ADY_MK_NARR_003.wav
https://cocoon.huma-num.fr/data/konuk/ADY_MK_NARR_003.mp3
https://cocoon.huma-num.fr/data/konuk/ADY_MK_NARR_003.xml
https://cocoon.huma-num.fr/data/konuk/masters/ADY_MK_NARR_00

/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_34305/482101636.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_exploded['wav'] = df_exploded['items_urls'].apply(extract_wav_urls_from_html)


In [160]:
df_exploded

,Href,Language,items_urls,wav
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/archi/masters...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/archi/masters...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/archi/masters...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/konuk/masters...
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/konuk/masters...
...,...,...,...,...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/fontaine/mast...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/fontaine/mast...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[https://cocoon.huma-num.fr/data/fontaine/mast...


In [161]:
df_exploded.to_csv('df_pangloss.csv')

In [156]:
df_exploded

,Href,Language,items_urls,wav
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
0,https://pangloss.cnrs.fr/corpus/Abzakh?lang=en,Abzakh,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
...,...,...,...,...
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]
250,https://pangloss.cnrs.fr/corpus/Yucuna?lang=en,Yucuna,https://pangloss.cnrs.fr/corpus/show?oai_prima...,[]


In [110]:
h = df['items_urls'].values[0][1]


In [111]:
h

'https://pangloss.cnrs.fr/corpus/show?oai_primary=cocoon-397bf64a-0319-38b8-b6c8-3e4a4acddb17&corpus=Abzakh'

In [114]:
tr = requests.get(h).text

In [115]:
tr

'\n\n<head>\n\t<link rel="icon" type="image/png" href="https://pangloss.cnrs.fr/favicon.png">\n\t<meta charset="utf-8">\n\t<meta name="viewport" content="width=device-width, initial-scale=1.0, minimum-scale=1.0">\n\t\t<title>Collection Pangloss | corpus abzakh - Patsitse le corbeau</title>\n\t<link rel="stylesheet" type="text/css" href="https://pangloss.cnrs.fr/dist/main.css?v=4">\n\t<script src="https://pangloss.cnrs.fr/dist/main.js"></script>\n\t<script>\n\t\tconst CURRENTLANGUAGE = "fr"\n\t</script>\n\t<script>\n\t\t// CONSTANTS\n\t\tconst HOMEURL = \'https://pangloss.cnrs.fr/\';\n\t\tconst APIURL = \'https://pangloss.cnrs.fr//api/\';\n\t\tconst ASSETSURL = \'https://pangloss.cnrs.fr//assets/\';\n\t\tconst MODULESURL = \'https://pangloss.cnrs.fr//modules/\';\n\t\tconst URLPARAMETERS = \'{"oai_primary":"cocoon-397bf64a-0319-38b8-b6c8-3e4a4acddb17","corpus":"Abzakh","oai_secondary":"cocoon-b316a847-9762-324e-9483-a386ee16bc92"}\';\n\t</script>\n</head><header class="m_header" id="site

In [116]:
tr = BeautifulSoup(tr, 'html.parser')

In [117]:
tr


<head>
<link href="https://pangloss.cnrs.fr/favicon.png" rel="icon" type="image/png"/>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0, minimum-scale=1.0" name="viewport"/>
<title>Collection Pangloss | corpus abzakh - Patsitse le corbeau</title>
<link href="https://pangloss.cnrs.fr/dist/main.css?v=4" rel="stylesheet" type="text/css"/>
<script src="https://pangloss.cnrs.fr/dist/main.js"></script>
<script>
		const CURRENTLANGUAGE = "fr"
	</script>
<script>
		// CONSTANTS
		const HOMEURL = 'https://pangloss.cnrs.fr/';
		const APIURL = 'https://pangloss.cnrs.fr//api/';
		const ASSETSURL = 'https://pangloss.cnrs.fr//assets/';
		const MODULESURL = 'https://pangloss.cnrs.fr//modules/';
		const URLPARAMETERS = '{"oai_primary":"cocoon-397bf64a-0319-38b8-b6c8-3e4a4acddb17","corpus":"Abzakh","oai_secondary":"cocoon-b316a847-9762-324e-9483-a386ee16bc92"}';
	</script>
</head><header class="m_header" id="site-header">
<div class="container">
<section class="logo-contain

In [124]:
def extract_wav_urls_from_html(html_content):
    # Parse the HTML
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the <ul> tag with class "download-list"
    ul_tag = soup.find('ul', class_='download-list')

    # Initialize a list to store the URLs ending with ".wav"
    wav_urls = []

    # Check if the <ul> tag is found
    if ul_tag:
        # Find all <a> tags within the <ul> tag
        a_tags = ul_tag.find_all('a', href=True)
        
        # Iterate through each <a> tag
        for a_tag in a_tags:
            href = a_tag['href']
            # Check if the href ends with ".wav"
            if href.endswith('.wav'):
                wav_urls.append(href)

    return wav_urls

In [127]:
ul_tag = tr.find('ul', class_='download-list')


In [128]:
ul_tag

<ul class="download-list">
<li><a download="" href="https://cocoon.huma-num.fr/data/archi/masters/125799.wav">Fichier média original (.wav)</a></li><li><a download="" href="https://cocoon.huma-num.fr/data/archi/mp3/125799_PATSITSE_44k.mp3">Fichier média compressé (.mp3)</a></li><li><a download="" href="https://cocoon.huma-num.fr/data/lacito/masters/crdo-ADY_PATSITSE.xml">Annotation 1 (.xml)</a></li> </ul>

In [130]:
wav_urls=[]
if ul_tag:
        # Find all <a> tags within the <ul> tag
        a_tags = ul_tag.find_all('a', href=True)
        for a_tag in a_tags:
            href = a_tag['href']
            # Check if the href ends with ".wav"
            if href.endswith('.wav'):
                wav_urls.append(href)

In [131]:
wav_urls

['https://cocoon.huma-num.fr/data/archi/masters/125799.wav']

In [126]:
a_tag

<a href="https://pangloss.cnrs.fr/">
<img alt="Pangloss" src="https://pangloss.cnrs.fr/assets/images/pangloss-logo.svg"/>
</a>

In [123]:
if a_tag:
    # Extract the href attribute
    url = a_tag['href']
    print(url)
else:
    print("URL not found.")


https://pangloss.cnrs.fr/
